# District of Columbia 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready ward-level table for District of Columbia, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). Note, there is no general election dataset for DC 2008 so far.

**Output**: A single CSV where each row is a ward and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- Party totals: `rep_primary_total`, `dem_primary_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [2]:
# DC 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/DC/20080212__dc__primary__president__ward.csv"
# GENERAL_PATH = r""

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/DC/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [3]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,ward,office,district,party,candidate,votes
0,1,President,NaN,DEM,BILL RICHARDSON,14
1,1,President,NaN,DEM,JOHN EDWARDS,36
2,1,President,NaN,DEM,DENNIS J. KUCINICH,40
3,1,President,NaN,DEM,HILLARY CLINTON,3941
4,1,President,NaN,DEM,BARACK OBAMA,10174
5,1,President,NaN,DEM,UNCOMMITTED,28
6,1,President,NaN,DEM,Write-ins,8
7,1,President,NaN,DEM,Total,14241
8,1,President,NaN,REP,MIKE HUCKABEE,72
9,1,President,NaN,REP,MITT ROMNEY,24


In [4]:
# Different values in 'office' column
primary_df["office"].value_counts()

office
President    112
Name: count, dtype: int64

In [5]:
# Number of missing values in each column
primary_df.isna().sum()

ward           0
office         0
district     112
party          0
candidate      0
votes          0
dtype: int64

In [6]:
# Now, drop the "office" column since it has only one value
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,ward,party,candidate,votes
0,1,DEM,BILL RICHARDSON,14
1,1,DEM,JOHN EDWARDS,36
2,1,DEM,DENNIS J. KUCINICH,40
3,1,DEM,HILLARY CLINTON,3941
4,1,DEM,BARACK OBAMA,10174
5,1,DEM,UNCOMMITTED,28
6,1,DEM,Write-ins,8
7,1,DEM,Total,14241
8,1,REP,MIKE HUCKABEE,72
9,1,REP,MITT ROMNEY,24


In [7]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Total                 16
BILL RICHARDSON        8
JOHN EDWARDS           8
DENNIS J. KUCINICH     8
HILLARY CLINTON        8
BARACK OBAMA           8
UNCOMMITTED            8
Write-ins              8
MIKE HUCKABEE          8
MITT ROMNEY            8
JOHN MCCAIN            8
RON PAUL               8
RUDY GIULIANI          8
Name: count, dtype: int64

Later on, we will recaculate the total by parties. Thus, for now, we can drop rows where `candidate == "Total"`.

In [10]:
# Drop rows where "candidate" == "Total"
primary_df = primary_df[primary_df["candidate"] != "Total"]
primary_df.head(DISPLAY_ROWS)

,ward,party,candidate,votes
0,1,DEM,BILL RICHARDSON,14
1,1,DEM,JOHN EDWARDS,36
2,1,DEM,DENNIS J. KUCINICH,40
3,1,DEM,HILLARY CLINTON,3941
4,1,DEM,BARACK OBAMA,10174
5,1,DEM,UNCOMMITTED,28
6,1,DEM,Write-ins,8
8,1,REP,MIKE HUCKABEE,72
9,1,REP,MITT ROMNEY,24
10,1,REP,JOHN MCCAIN,337


In [11]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
DEM    56
REP    40
Name: count, dtype: int64

In [12]:
# Data type of each column in primary_df
primary_df.dtypes

ward          int64
party        object
candidate    object
votes         int64
dtype: object

In [13]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,ward,party,candidate,votes
0,1,DEM,BILL RICHARDSON,14
1,1,DEM,JOHN EDWARDS,36
2,1,DEM,DENNIS J. KUCINICH,40
3,1,DEM,HILLARY CLINTON,3941
4,1,DEM,BARACK OBAMA,10174
5,1,DEM,UNCOMMITTED,28
6,1,DEM,Write-ins,8
8,1,REP,MIKE HUCKABEE,72
9,1,REP,MITT ROMNEY,24
10,1,REP,JOHN MCCAIN,337


In [14]:
# Shape after preprocessing
primary_df.shape

(96, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [15]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Lowercase party names
    """
    return(s.str.lower())

In [16]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [17]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [19]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri", key_col="ward")
primary_pivot.head(DISPLAY_ROWS)

,ward,pri_dem_CLINTON,pri_dem_EDWARDS,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNCOMMITTED,pri_dem_WRITEINS,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY
0,1,3941,36,40,10174,14,28,8,12,72,337,68,24
1,2,4410,56,19,7640,22,26,18,16,161,896,91,70
2,3,6452,73,38,10917,44,37,32,23,221,1490,137,140
3,4,3947,40,30,16469,12,58,13,8,110,333,30,20
4,5,2748,33,32,14270,16,51,11,4,90,144,32,19
5,6,4221,62,19,11332,21,57,11,31,270,903,119,106
6,7,2277,32,10,13578,7,57,14,3,61,58,13,8
7,8,1473,15,5,9003,9,25,7,4,35,37,4,11


Note that there are columns with uncommitted candidates or write-ins that still had votes (`pri_dem_UNCOMMITTED`, `pri_dem_WRITEINS`). We will keep this for total counting purposes and drop it at the end.

In [20]:
# Primary dataframe shape after pivot
primary_pivot.shape

(8, 13)

## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_primary_total` = sum of all `pri_rep_*` columns
* `dem_primary_total` = sum of all `pri_dem_*` columns

In [22]:
# Add party totals for primary election
rep_primary_cols   = [c for c in primary_pivot.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in primary_pivot.columns if c.startswith("pri_dem_")]

primary_pivot["rep_primary_total"] = primary_pivot[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
primary_pivot["dem_primary_total"] = primary_pivot[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the UNCOMMITTED and WRITEINS columns.

In [24]:
# Drop UNCOMMITTED and WRITEINS column for primary election
primary_pivot = primary_pivot.drop(columns=["pri_dem_UNCOMMITTED", "pri_dem_WRITEINS"])

# Snippet at the merged dataframe with primary totals
primary_pivot.head(DISPLAY_ROWS)

,ward,pri_dem_CLINTON,pri_dem_EDWARDS,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,rep_primary_total,dem_primary_total
0,1,3941,36,40,10174,14,12,72,337,68,24,513,14241
1,2,4410,56,19,7640,22,16,161,896,91,70,1234,12191
2,3,6452,73,38,10917,44,23,221,1490,137,140,2011,17593
3,4,3947,40,30,16469,12,8,110,333,30,20,501,20569
4,5,2748,33,32,14270,16,4,90,144,32,19,289,17161
5,6,4221,62,19,11332,21,31,270,903,119,106,1429,15723
6,7,2277,32,10,13578,7,3,61,58,13,8,143,15975
7,8,1473,15,5,9003,9,4,35,37,4,11,91,10537


In [26]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned primary dataframe:")
primary_pivot.columns

Final columns in the cleaned primary dataframe:


Index(['ward', 'pri_dem_CLINTON', 'pri_dem_EDWARDS', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_rep_GIULIANI',
       'pri_rep_HUCKABEE', 'pri_rep_MCCAIN', 'pri_rep_PAUL', 'pri_rep_ROMNEY',
       'rep_primary_total', 'dem_primary_total'],
      dtype='object')

In [27]:
# Preview the primary_pivot dataframe with totals
primary_pivot.head(DISPLAY_ROWS)

,ward,pri_dem_CLINTON,pri_dem_EDWARDS,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,rep_primary_total,dem_primary_total
0,1,3941,36,40,10174,14,12,72,337,68,24,513,14241
1,2,4410,56,19,7640,22,16,161,896,91,70,1234,12191
2,3,6452,73,38,10917,44,23,221,1490,137,140,2011,17593
3,4,3947,40,30,16469,12,8,110,333,30,20,501,20569
4,5,2748,33,32,14270,16,4,90,144,32,19,289,17161
5,6,4221,62,19,11332,21,31,270,903,119,106,1429,15723
6,7,2277,32,10,13578,7,3,61,58,13,8,143,15975
7,8,1473,15,5,9003,9,4,35,37,4,11,91,10537


Now, we save the cleaned dataframe into the processed directory.

In [28]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
primary_pivot.to_csv(OUTPUT_PATH + "DC.csv", index=False)